In [1]:
import pandas as pd
import numpy as np
import os
import json

In [2]:
# import os

# path = 'C:/Users/RSHIRINI/OneDrive - United Nations/Desktop/DSS/DATA COLLECTOR/datacollector_received_quest/poverty'
# parts = path.split('/')
# current = ""

# for part in parts:
#     current += part + "/"
#     if not os.path.exists(current):
#         print(f"❌ Path broken at: {current}")
#         break
#     else:
#         print(f"✅ Found: {current}")

In [3]:
# folder_path='C:/Users/RSHIRINI/OneDrive - United Nations/Desktop/DSS/DATA COLLECTOR/datacollector_received_quest/recieved_quests'

#for poverty
folder_path='C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/DATA COLLECTOR/datacollector_received_quest/poverty'
# folder_path='C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/DATA COLLECTOR'
#check all xlsx files in the folder

# Get all excel files in the directory
xlsx_files = [f for f in os.listdir(folder_path) if f.endswith('.xlsx')]

xlsx_files

['exports_Tunisia poverty_20260505_074030.xlsx']

In [ ]:
dataframes = []

for file in xlsx_files:
    print(f'--- Opening Workbook: {file} ---')
    full_path = os.path.join(folder_path, file)
    
    # Load the workbook once to access its sheet names
    xls = pd.ExcelFile(full_path)
    
    for sheet_name in xls.sheet_names:
        print(f'  Processing sheet: {sheet_name}')
        
        # Read the specific sheet
        df_raw = pd.read_excel(xls, sheet_name=sheet_name, header=None, dtype=str)

        #gets the indices where text 'index' is in the sheet
        header_rows = df_raw.index[df_raw[0] == 'index'].tolist()
        if len(header_rows) < 2:
            raise ValueError(
                f"\n\nERROR: Structure mismatch in File: '{file}', Sheet: '{sheet_name}'.\n"
            )

        # get the headers
        header_1_values = df_raw.iloc[header_rows[0]].dropna()
        header_2_values = df_raw.iloc[header_rows[1]].dropna()

        # Data Slice (index=1)
        df_data = df_raw[df_raw[0] == '1'].copy()
        df_data.columns = header_1_values
        #get the id_vars for merging later. that is get all non digit (the years) columns
        id_variables = [col for col in header_1_values if not col.isdigit()]

        df_data = df_data.melt(id_vars=id_variables, var_name='السنة', value_name='العدد')
        if 'index' in df_data.columns:
            df_data.drop(columns=['index'], inplace=True)

        # Source Slice (index=2)
        df_source = df_raw[df_raw[0] == '2'].copy()
        df_source = df_source[header_2_values.index]
        df_source.columns = header_2_values
        if 'index' in df_source.columns:
            df_source.drop(columns=['index'], inplace=True)

        # Merge
        df_merged = pd.merge(df_data, df_source, on=['السنة', 'المؤشر', 'الدولة'], how='left')
        
        # Track both file and sheet for transparency
        df_merged['sheet_source'] = sheet_name
        
        dataframes.append(df_merged)

# Final aggregation
if dataframes:
    final_df = pd.concat(dataframes, ignore_index=True)
    final_df = final_df.dropna(axis=1, how='all')
    final_df.to_excel('final_dataset_combined.xlsx', index=False)
    print("\nSuccess: Combined data from all workbooks and sheets.")
else:
    print("\nNo data found. Check your folder path and sheet structures.")

--- Opening Workbook: exports_Tunisia poverty_20260505_074030.xlsx ---
  Processing sheet: Tunisia_poverty - Poverty_5
  Processing sheet: Tunisia_poverty - Poverty_1
  Processing sheet: Tunisia_poverty - Poverty_4
  Processing sheet: Tunisia_poverty - Poverty_3
  Processing sheet: Tunisia_poverty - Poverty_2

Success: Combined data from all workbooks and sheets.


### check unique values in columns

In [ ]:
import difflib

# Keep THRESHOLD very high to only catch typos/spaces, not different indicators
THRESHOLD = 0.98 
fuzzy_report = {}

# Compare each value with every other value in the list
'''enumerate(values): This gives you two things at once: the index i (the position) and the val1 (the actual text, like "الجزائر").
This loop picks the "Anchor" item. We are going to hold this item in our hand and compare it to others.'''
 
for column, values in unique_values_dict.items():
    similar_pairs = []
    
    for i, val1 in enumerate(values):
        for val2 in values[i+1:]:
            # Use raw strings to catch hidden spaces
            s1, s2 = str(val1), str(val2)
            score = difflib.SequenceMatcher(None, s1, s2).ratio()
            
            #Only show if they are ALMOST identical but NOT exactly 1.0
            if score >= THRESHOLD and score < 1.0:
                # Get the sources
                src1 = final_df[final_df[column] == val1]['file_source'].unique().tolist()
                src2 = final_df[final_df[column] == val2]['file_source'].unique().tolist()
                
                # Format exactly as you requested
                # Wrapping values in quotes "" helps you see the trailing spaces in the terminal
                report_entry = f"\"{val1}\" {src1} <<<--->>> \"{val2}\" {src2}"
                similar_pairs.append(report_entry)
                
    if similar_pairs:
        fuzzy_report[column] = similar_pairs

# Print the diagnostic report
print(json.dumps(fuzzy_report, indent=4, ensure_ascii=False))


{
    "المؤشر": [
        "\"النسبة المئوية (%)  للتحصين ضد الثلاثي (الكزاز+ الدفتريا + السعال الديكي)، للأطفال الذين تتراوح أعمارهم بين 12-23 شهرا\" ['algeria - Health_5_a.xlsx', 'comoros - Health_5_a.xlsx', 'djibouti - Health_5_a.xlsx', 'iraq - Health_5_a.xlsx', 'jordan - Health_5_a.xlsx', 'lebanon - Health_5_a.xlsx', 'libya - Health_5_a.xlsx', 'mauritania - Health_5_a.xlsx', 'morocco - Health_5_a.xlsx', 'oman - Health_5_a.xlsx', 'palestine - Health_5_a.xlsx', 'qatar - Health_5_a.xlsx', 'saudi - Health_5_a.xlsx', 'somalia - Health_5_a.xlsx', 'sudan - Health_5_a.xlsx', 'syria - Health_5_a.xlsx', 'tunisia - Health_5_a.xlsx', 'uae - Health_5_a.xlsx', 'yemen - Health_5_a.xlsx'] <<<--->>> \"النسبة المئوية (%)  للتحصين ضد الثلاثي (الكزاز+ الدفتريا + السعال الديكي)، للأطفال الذين تتراوح أعمارهم بين 12-23 شهراً\" ['bahrain - Health_5_a.xlsx', 'egypt - Health_5_a.xlsx', 'kuwait - Health_5_a.xlsx']",
        "\"النسبة المئوية (%)  للتحصين ضد الحصبة للأطفال الذين تتراوح أعمارهم بين 12-23 شهرا\"